### **Loading Data**

In [25]:
from pathlib import Path
import pandas as pd

DATA = "reviews.csv"
path = next((p for p in (Path(DATA), Path("data") / DATA) if p.exists()), None)
if path is None:
    from google.colab import files
    files.upload()
    path = Path(DATA)

df = pd.read_csv(path)
print(df.shape)
df.head()

(3000, 3)


,source,label,sentence
0,amazon,0,So there is no way for me to plug it in here i...
1,amazon,1,"Good case, Excellent value."
2,amazon,1,Great for the jawbone.
3,amazon,0,Tied to charger for conversations lasting more...
4,amazon,1,The mic is great.


In [26]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

print("pandas", pd.__version__, "| NumPy", np.__version__, "| matplotlib", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)

pandas 2.2.3 | NumPy 2.1.3 | matplotlib 3.10.0
scikit-learn 1.6.1


In [27]:
import re, time
from collections import Counter, defaultdict

import nltk
import spacy

for resource in ("punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"):
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

try:
    nlp_spacy = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp_spacy = spacy.load("en_core_web_sm")

print("NLTK", nltk.__version__, "| spaCy", spacy.__version__, "| pandas", pd.__version__)

NLTK 3.9.1 | spaCy 3.8.16 | pandas 2.2.3


In [28]:
counts = df["label"].value_counts()
proportions = (counts / len(df) * 100).round(1)
pd.DataFrame({"sentences": counts, "proportion (%)": proportions})

,sentences,proportion (%)
label,,
0,1500,50.0
1,1500,50.0


imbalance affects accuracy...

In [29]:
rest, test_rows = train_test_split(df.index, test_size=0.15, stratify=df["label"], random_state=42)
train_rows, val_rows = train_test_split(rest, test_size=len(test_rows),
                                        stratify=df.loc[rest, "label"], random_state=42)

split = pd.Series("train", index=df.index)
split[val_rows] = "validation"
split[test_rows] = "test"
pd.crosstab(split, df["label"], margins=True)

label,0,1,All
row_0,,,
test,225,225,450
train,1050,1050,2100
validation,225,225,450
All,1500,1500,3000


In [30]:
train, val, test = df.loc[train_rows], df.loc[val_rows], df.loc[test_rows]
X_train_text, y_train = train["sentence"], train["label"]
X_val_text, y_val = val["sentence"], val["label"]
X_test_text, y_test = test["sentence"], test["label"]
print(len(train), len(val), len(test))

2100 450 450


### **Preprocessing**

In [31]:
STOP = set(stopwords.words("english"))
porter, wordnet = PorterStemmer(), WordNetLemmatizer()

def preprocess(text, lowercase=True, alpha_only=True, stop_words=STOP, keep=(), reduce=None, expand=False):
    if expand:
        text = re.sub(r"n't\b", " not", text)
    tokens = word_tokenize(text)
    if lowercase:
        tokens = [t.lower() for t in tokens]
    if alpha_only:
        tokens = [t for t in tokens if t.isalpha()]
    if stop_words:
        tokens = [t for t in tokens if t not in stop_words or t in keep]
    if reduce == "stem":
        tokens = [porter.stem(t) for t in tokens]
    elif reduce == "lemma":
        tokens = [t.lemma_.lower() for t in nlp_spacy(" ".join(tokens))]
    return tokens

def stage_table(texts, lowercase=True, alpha_only=True, stop_words=STOP, keep=(), reduce=None, expand=False):
    if expand:
        texts = [re.sub(r"n't\b", " not", text) for text in texts]
    rows, tokens = [], [t for text in texts for t in word_tokenize(text)]
    rows.append(("Tokens" + (" (contractions expanded)" if expand else ""), tokens))
    if lowercase:
        tokens = [t.lower() for t in tokens]
        rows.append(("Lowercased", tokens))
    if alpha_only:
        tokens = [t for t in tokens if t.isalpha()]
        rows.append(("Alphabetic only", tokens))
    if stop_words:
        tokens = [t for t in tokens if t not in stop_words or t in keep]
        rows.append(("Stop words removed" + (" (negation kept)" if keep else ""), tokens))
    if reduce == "stem":
        tokens = [porter.stem(t) for t in tokens]
        rows.append(("Porter stems", tokens))
    elif reduce == "lemma":
        partial = dict(lowercase=lowercase, alpha_only=alpha_only, stop_words=stop_words, keep=keep)
        tokens = [t.lemma_.lower() for doc in nlp_spacy.pipe((" ".join(preprocess(x, **partial)) for x in texts), batch_size=500) for t in doc]
        rows.append(("spaCy lemmas", tokens))
    return pd.DataFrame([(n, len(t), len(set(t))) for n, t in rows], columns=["stage", "tokens", "types"]).set_index("stage")

def show(texts, ids, **params):
    for i in ids:
        print(f"[{i}] {texts[i]}")
        print(f"     -> {' '.join(preprocess(texts[i], **params))}")

In [32]:
NEGATION = {"not", "n't", "no", "never", "nor", "nothing", "neither", "cannot", "without", "hardly"}

def has_negation(tokens):
    return any(t in NEGATION for t in tokens)

before = df["sentence"].apply(lambda s: has_negation(word_tokenize(s.lower())))
after = df["sentence"].apply(lambda s: has_negation(preprocess(s)))
print(f"sentences with a negation word: {before.sum()} ({before.mean():.0%} of the corpus)")
print(f"of which negative: {df[before]['label'].eq(0).sum()}, positive: {df[before]['label'].eq(1).sum()}")
print(f"still containing one after the default pipeline: {after.sum()}")

sentences with a negation word: 691 (23% of the corpus)
of which negative: 532, positive: 159
still containing one after the default pipeline: 93


# **Modeling**

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_val)
print("training accuracy:", round(accuracy_score(y_train, lr.predict(X_train)), 3))
print("validation accuracy:", round(accuracy_score(y_val, lr_pred), 3))